# ಪಾಠ 11 - ಮಾದರಿ ಸಂದರ್ಭ ಪ್ರೋಟೋಕಾಲ್ (MCP)

The **ಮಾದರಿ ಸಂದರ್ಭ ಪ್ರೋಟೋಕಾಲ್ (MCP)** ಒಂದು ಓಪನ್ ಸ್ಟಾಂಡರ್ಡ್ ಆಗಿದ್ದು, ಏಜೆಂಟ್‌ಗಳಿಗೂ ರನ್‌ಟೈಮ್‌ನಲ್ಲಿ ಸಾಧನಗಳು, ಸಂಪನ್ಮೂಲಗಳು ಮತ್ತು ಡೇಟಾ ಮೂಲಗಳನ್ನು ಡೈನಾಮಿಕ್ ಆಗಿ ಕಂಡುಹಿಡಿಯಲು ಮತ್ತು ಬಳಸಿಕೊಳ್ಳಲು ಸಾಧ್ಯವಾಗಿಸುತ್ತದೆ. ಏಜೆಂಟ್‌ಗೆ ಸಾಧನಗಳನ್ನು ಹಾರ್ಡ್‌ಕೋಡ್ ಮಾಡುವ ಬದಲು, MCP ಏಜೆಂಟ್‌ಗಳಿಗೆ ಬೇಡಿಕೆಯಂತೆ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ಬಹಿರಂಗಪಡಿಸುವ ಹೊರಗಿನ ಸರ್ವರ್‌ಗಳಿಗೆ ಸಂಪರ್ಕಿಸಲು ಅವಕಾಶ ನೀಡುತ್ತದೆ.

ಈ ಪಾಠದಲ್ಲಿ, ನೀವು ಕಲಿಯುತ್ತೀರಿ:
- MCP ಏನು ಮತ್ತು ಅದು ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳಿಗಾಗಿ ಏಕೆ ಮಹತ್ವಪೂರ್ಣವಾಗಿದೆ
- MCP ಕ್ಲೈಂಟ್-ಸರ್ವರ್ ಆರ್ಕಿಟೆಕ್ಚರ್ ಹೇಗೆ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತದೆ
- MCP ಶೈಲಿಯ ಸಾಧನ ಅನ್ವೇಷಣೆ ಬಳಸುವ ಏಜೆಂಟ್‌ಗಳನ್ನು ಹೇಗೆ ನಿರ್ಮಿಸಬೇಕು


## ಸಿದ್ಧತೆ

**ಆವಶ್ಯಕತೆಗಳು:**
- ಡಿಪ್ಲಾಯ್ ಮಾಡಲಾದ ಮಾದರಿಯನ್ನು ಹೊಂದಿರುವ Azure AI Foundry ಪ್ರಾಜೆಕ್ಟ್
- ಪ್ರಾಮಾಣೀಕರಣಕ್ಕಾಗಿ `az login` ಅನ್ನು ಚಲಾಯಿಸಿ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Model Context Protocol (MCP) ಎಂದರೆ ಏನು?

MCP ಎಐ ಏಜೆಂಟ್‌ಗಳಿಗೆ ಬಾಹ್ಯ ಉಪಕರಣಗಳು ಮತ್ತು ದತ್ತಾ ಮೂಲಗಳನ್ನು ಪತ್ತೆ ಹಚ್ಚಿ ಅವುಗಳೊಂದಿಗೆ ಸಂವಹನ ಮಾಡಲು ಮಾನಕ ಮಾರ್ಗವನ್ನು ನಿರ್ಧರಿಸುತ್ತದೆ:

- **MCP Server**: ಮಾನಕ ಪ್ರೋಟೋಕಾಲ್ ಮೂಲಕ ಉಪಕರಣಗಳು, ಸಂಪನ್ಮೂಲಗಳು ಮತ್ತು ಪ್ರಾಂಪ್ಟ್‌ಗಳನ್ನು ಪ್ರಕಟಿಸುತ್ತದೆ
- **MCP Client**: ಸರ್ವರ್‌lere ಸಂಪರ್ಕಿಸಿ ಲಭ್ಯವಿರುವ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ಪತ್ತೆಹಚ್ಚುವ ಏಜೆಂಟ್ ರನ್‌ಟೈಮ್
- **Dynamic Discovery**: ಏಜೆಂಟ್‌ಗಳಿಗೆ ಹಾರ್ಡ್‌ಕೋಡ್ ಮಾಡಿದ ಉಪಕರಣಗಳ ಅಗತ್ಯವಿಲ್ಲ — ಅವು ರನ್‌ಟೈಮ್‌ನಲ್ಲಿ ಲಭ್ಯವಿರುವುದನ್ನು ಪತ್ತೆಹಚ್ಚುತ್ತವೆ

ಇದು ವಿಸ್ತರಿಸಬಹುದಾದ ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳನ್ನು ನಿರ್ಮಿಸುವಾಗ ಪರಿಣಾಮಕಾರಿಯಾಗಿದೆ, ಅಲ್ಲಿ ಹೊಸ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ಏಜೆಂಟ್ ಕೋಡ್ ಅನ್ನು ಬದಲಾಯಿಸದೆ ಸೇರಿಸಬಹುದು.


## MCP ಹೇಗೆ ಕೆಲಸ ಮಾಡುತ್ತದೆ

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ಏಜೆಂಟ್ (MCP client) MCP ಸರ್ವರ್‌ಗೆ ಸಂಪರ್ಕಗೊಳ್ಳುತ್ತದೆ
2. ಸರ್ವರ್ ಲಭ್ಯವಿರುವ ಉಪಕರಣಗಳ ಮತ್ತು ಅವುಗಳ ಸ್ಕೀಮಾಗಳ ಪಟ್ಟಿಯನ್ನು ನೀಡುತ್ತದೆ
3. ನಂತರ ಏಜೆಂಟ್ ತನ್ನ ತರ್ಕಿಸುವ ವೇಳೆ ಕಂಡುಬಂದ ಯಾವುದೇ ಉಪಕರಣವನ್ನು ಕರೆಮಾಡಬಹುದು
4. ಫಲಿತಾಂಶಗಳು ಅದೇ ಪ್ರೋಟೋಕಾಲ್ ಮೂಲಕ ಹಿಂತಿರುಗುತ್ತವೆ


## MCP ಉಪಕರಣಗಳ ಕಂಡುಹಿಡಿತವನ್ನು ಅನುಕರಿಸುವುದು

ವಾಸ್ತವ MCP ಸರ್ವರ್ ಚಾಲನೆಯಲ್ಲಿರುವ ಸರ್ವರ್ ಪ್ರಕ್ರಿಯೆಯನ್ನು ಅಗತ್ಯವಿರುವುದರಿಂದ, ನಾವು `@tool` ಕಾರ್ಯಗಳನ್ನು ಬಳಸಿ MCP-ಸಂಪರ್ಕಿತ ವಸತಿ ಸೇವೆ ಏನಾಗಿರಬಹುದೆಂದು ಅನುಕರಿಸುವ ಮಾದರಿಯನ್ನು ತೋರಿಸುತ್ತೇವೆ.

ಉತ್ಪಾದನೆಯಲ್ಲಿ, ಈ ಉಪಕರಣಗಳನ್ನು ಸ್ಥಳೀಯವಾಗಿ ವ್ಯಾಖ್ಯಾನಿಸುವ ಬದಲು MCP ಸರ್ವರ್‌ನಿಂದ ಡೈನಾಮಿಕ್‌ವಾಗಿ ಕಂಡುಹಿಡಿಯಲಾಗುತ್ತದೆ.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-ಶೈಲಿಯ ಉಪಕರಣಗಳೊಂದಿಗೆ ಏಜೆಂಟ್ ನಿರ್ಮಿಸುವುದು


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## ಉತ್ಪಾದನೆಯಲ್ಲಿ MCP

In a production environment, MCP enables powerful patterns:

- **ಡೈನಾಮಿಕ್ ಉಪಕರಣ ಕಂಡುಹಿಡಿಯುವಿಕೆ**: ಏಜೆಂಟುಗಳು MCP ಸರ್ವರ್‌ಗಳಿಗೆ ಸಂಪರ್ಕಿಸಿ ರನ್‌ಟೈಮ್‌ನಲ್ಲಿ ಉಪಕರಣಗಳನ್ನು ಕಂಡುಹಿಡಿಯುತ್ತವೆ
- **ವಿಚ್ಛೇದಿತ ವಾಸ್ತುಶಿಲ್ಪ**: ಉಪಕರಣದ ಒದಗಿಸುವವರು ಏಜೆಂಟಿನಿಂದ ಸ್ವತಂತ್ರವಾಗಿ ನವೀಕರಣಗಳನ್ನು ಮಾಡಬಹುದು
- **ಸಂಸ್ಥೆಗಳ ನಡುವಿನ ಹಂಚಿಕೆ**: ತಂಡಗಳು MCP ಸರ್ವರ್‌ಗಳ ಮೂಲಕ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ಬಹಿರಂಗಗೊಳಿಸಬಹುದು ಮತ್ತು ಯಾವುದೇ ಏಜೆಂಟ್ ಅವನ್ನು ಬಳಸಬಹುದು
- **Microsoft Agent Framework ಬೆಂಬಲ**: MAF ನಲ್ಲಿ ರೂಢಗೊಂಡ MCP ಕ್ಲೈಂಟ್ ಬೆಂಬಲ `mcp` ಇಂಟಿಗ್ರೇಷನ್ ಮೂಲಕ ಲಭ್ಯವಿದೆ

To use a real MCP server with MAF, you would connect via `hosted_mcp_tool()` or the MCP client integration.

**ಹೆಚ್ಚು ತಿಳಿದುಕೊಳ್ಳಿ:**
- [MCP ವಿಶೇಷಣ](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP ಬೆಂಬಲ](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ, ನೀವು ಕಲಿತಿರಿ:
- **MCP** ಏಜೆಂಟ್‌ಗಳು ಮತ್ತು ಉಪಕರಣ ಒದಗಿಸುವವರ ನಡುವೆ ಗತಿಶೀಲ ಉಪಕರಣ ಪತ್ತೆಗಾಗಿ ತೆರೆಯಲಾದ ಮಾನಕವಾಗಿದೆ
- **ಗ್ರಾಹಕ-ಸರ್ವರ್ ವಾಸ್ತುಶಿಲ್ಪ** ಏಜೆಂಟ್‌ಗಳಿಗೆ ರನ್‌ಟೈಮ್‌ನಲ್ಲಿ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ಪತ್ತೆಹಚ್ಚಲು ಅವಕಾಶ ನೀಡುತ್ತದೆ
- **MCP** ಕೋಡ್ ಬದಲಾವಣೆಗಳಿಲ್ಲದೆ ಉಪಕರಣಗಳನ್ನು ಸೇರಿಸಲು ಸಾಧ್ಯವಿರುವ **ವಿಸ್ತರಿಸಬಹುದಾದ, ಪ್ರತ್ಯೇಕಗೊಂಡ ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳನ್ನು** ಸಕ್ರಿಯಗೊಳಿಸುತ್ತದೆ
- Microsoft Agent Framework ಉತ್ಪಾದನಾ ಬಳಕೆಗೆ **ಒಳಗೊಂಡಿರುವ MCP ಬೆಂಬಲವನ್ನು** ಒದಗಿಸುತ್ತದೆ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ನಿರಾಕರಣೆ**:
ಈ ದಾಖಲೆವನ್ನು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ಶುದ್ಧತೆಯ ಹಿನ್ನೆಲೆ ಪ್ರಯತ್ನಿಸಿದರೂ ಸಹ ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಮರ್ಪಕತೆಗಳು ಇರಬಹುದು ಎಂದು ದಯವಿಟ್ಟು ಗಮನದಲ್ಲಿರಿಸಿ. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ದಾಖಲೆವನ್ನೇ ಅಧಿಕೃತ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಅಥವಾ ನಿರ್ಣಾಯಕ ಮಾಹಿತಿಗಾಗಿ ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದದ ಬಳಕೆಯಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ಅಸಮanjasoತೆಗಳು ಅಥವಾ ತಪ್ಪಾಗಿ ವ್ಯಾಖ್ಯಾನಗೊಳ್ಳುವ ಪರಿಣಾಮಗಳಿಗಾಗಿ ನಾವು ಜವಾಬ್ದಾರಿಗಳಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
